In [80]:
from haystack_integrations.components.generators.ollama import OllamaChatGenerator

chat_generator = OllamaChatGenerator(
    model="qwen2.5:7b",
    url="http://localhost:11434",
    timeout=300,
    generation_kwargs={
        "num_predict": 3000,
        "temperature": 0.2,
    },
)

In [81]:
from dataclasses import dataclass, field
from typing import Callable
import requests

import os
import requests
from dotenv import load_dotenv

from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function
from haystack.components.tools import ToolInvoker

load_dotenv()

True

In [82]:
from dataclasses import dataclass, field
from typing import Callable, List

from haystack.dataclasses import ChatMessage
from haystack.tools import create_tool_from_function
from haystack.components.tools import ToolInvoker


@dataclass
class ToolCallingAgent:
    name: str = "ToolCallingAgent"
    llm: object = None

    instructions: str = """
You are a STRICT research agent.

RULES:
1. ALWAYS call web_search for ANY question about papers, research, or concepts.
2. NEVER answer from memory.
3. NEVER say "I couldn't find".
4. If tool results exist, you MUST use them.
5. Extract ALL papers from tool output.
6. Format:

1. Paper Title
2. Link
3. 2–3 sentence summary
4. Why it is relevant

If multiple papers exist → include ALL.
Be factual. No guessing.
"""

    functions: List[Callable] = field(default_factory=list)

    def __post_init__(self):
        self._system_message = ChatMessage.from_system(self.instructions)

        # convert functions into tools
        self.tools = [
            create_tool_from_function(fun)
            for fun in self.functions
        ]

        # tool executor
        self._tool_invoker = ToolInvoker(
            tools=self.tools,
            raise_on_failure=False
        )

    def run(self, messages):

        # STEP 0: add system prompt
        messages = [self._system_message] + messages

        # STEP 1: LLM call (decide tool or answer)
        response = self.llm.run(messages=messages, tools=self.tools)
        msg = response["replies"][0]

        messages.append(msg)

        # STEP 2: TOOL EXECUTION
        if msg.tool_calls:
            tool_results = self._tool_invoker.run(messages=[msg])["tool_messages"]
            messages.extend(tool_results)

            # STEP 3: SECOND LLM CALL (IMPORTANT)
            final_response = self.llm.run(messages=messages)["replies"][0]
            return [final_response]

        # no tool call → return direct answer
        return [msg]

In [83]:
import requests


class TavilyWebSearch:
    def __init__(self, api_key: str, top_k: int = 3):
        self.api_key = api_key
        self.top_k = top_k

    def run(self, query: str):
        payload = {
            "api_key": self.api_key,
            "query": query,
            "max_results": self.top_k,
            "include_answer": True,
            "include_domains": [
                "arxiv.org",
                "openreview.net",
                "aclanthology.org",
                "ieee.org",
                "springer.com"
            ],
            "exclude_domains": [
                "wikipedia.org",
                "reddit.com"
            ]
        }

        try:
            resp = requests.post(
                "https://api.tavily.com/search",
                json=payload,
                timeout=15,
            )
            resp.raise_for_status()
            data = resp.json()

        except requests.RequestException as e:
            return {
                "answer": None,
                "results": [],
                "error": str(e)
            }

        results = []

        for hit in data.get("results", []):
            results.append({
                "title": hit.get("title", ""),
                "url": hit.get("url", ""),
                "content": (hit.get("content", "") or "")[:300]
            })

        return {
            "answer": data.get("answer", ""),
            "results": results
        }

In [84]:
tavily = TavilyWebSearch(
    api_key=os.environ["TAVILY_API_KEY"],
    top_k=3
)

def web_search(query: str):
    result = tavily.run(query=query)

    if isinstance(result, str):
        return {"results": [], "answer": result}

    return {
        "answer": result.get("answer"),
        "results": [
            {
                "title": r.get("title"),
                "url": r.get("url"),
                "content": r.get("content", "")
            }
            for r in result.get("results", [])
        ]
    }

In [ ]:
import requests

def download_pdf(url: str, filename: str = "paper.pdf") -> str:
    try:
        response = requests.get(
            url,
            timeout=20,
            headers={"User-Agent": "Mozilla/5.0"}
        )

        content_type = response.headers.get("Content-Type", "")

        if "pdf" not in content_type.lower():
            return "Not a PDF link"

        with open(filename, "wb") as f:
            f.write(response.content)

        return f"Saved PDF as {filename}"

    except Exception as e:
        return f"Download failed: {str(e)}"

In [98]:
def ask_permission(url: str) -> bool:
    answer = input(f"\nDownload this paper?\n{url}\n(y/n): ")
    return answer.lower() in ["y", "yes"]

In [99]:
def download_paper(url: str):
    if not ask_permission(url):
        return "User rejected download"

    return download_pdf(url)

In [86]:
import re

def extract_citations(text: str):
    results = []

    # split into sentences
    sentences = re.split(r'(?<=[.!?])\s+', text)

    for s in sentences:
        citations = re.findall(r"\[\d+\]", s)

        if citations:
            results.append({
                "thesis": s,
                "citations": citations
            })

    return results

In [87]:
def add(a: int, b: int) -> int:
    return a + b

In [ ]:
agent = ToolCallingAgent(
    llm=chat_generator,
    functions=[web_search, add, download_paper]
)

In [89]:
messages = [
    ChatMessage.from_user("What is the Paper for agentic AI?")
]

In [90]:
response = agent.run(messages)

d:\dev\Lena\dat_project\dat_venv\Lib\site-packages\haystack_integrations\components\generators\ollama\chat\chat_generator.py:178: Warning: Mutating attribute '_meta' on an instance of 'ChatMessage' can lead to unexpected behavior by affecting other parts of the pipeline that use the same dataclass instance. Use `dataclasses.replace(instance, _meta=new_value)` instead. See https://docs.haystack.deepset.ai/docs/custom-components#requirements for details.
  chat_msg._meta = _convert_ollama_meta_to_openai_format(response_dict)


In [93]:
for msg in response:
    print(msg.text)

Here are the key papers related to agentic AI:

1. **Paper Title**: Agentic AI Security: Threats, Defenses, Evaluation, and Open ...
   - **Link**: [https://arxiv.org/html/2510.23883v1](https://arxiv.org/html/2510.23883v1)
   - **Summary**: This paper highlights the potential of agentic AI in accelerating scientific progress and expanding the knowledge frontier.
   - **Relevance**: It addresses security aspects, which are crucial for the responsible development and deployment of agentic AI systems.

2. **Paper Title**: Agentic AI: A Comprehensive Survey of Architectures, Applications, and Future Directions
   - **Link**: [https://arxiv.org/abs/2510.25445](https://arxiv.org/abs/2510.25445)
   - **Summary**: This comprehensive survey covers the architectures, applications, and future directions of agentic AI.
   - **Relevance**: It provides a broad overview of the field, making it essential for understanding the current state and potential developments in agentic AI.

3. **Paper Title**: